# hw 507
請比較Learning Regression及CART regression Tree模型預測boston房價的效能。
- 計算模型的訓練資料集與測試資料集的$R^2$
- 比較Linear Regression與CART regression Tree模型的測試資料集的效能($R^2$)

In [20]:
#導入所需的套件
import pandas as pd # 導入 pandas 庫，用於數據處理和分析
import numpy as np # 導入 numpy 庫，用於數值計算
import matplotlib.pyplot as plt # 導入 matplotlib 庫，用於數據可視化

In [21]:
#讀取boston資料
import kagglehub

# 從 Kaggle 下載最新版本的 boston-housing-dataset
path = kagglehub.dataset_download("altavish/boston-housing-dataset")

# 讀取下載的 CSV 檔案到 DataFrame 中
df = pd.read_csv(f"{path}/HousingData.csv")
df.head() # 顯示資料框的前五行，初步查看資料內容

Using Colab cache for faster access to the 'boston-housing-dataset' dataset.


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,NaN,36.2


# 作業開始

### 資料先處理

In [22]:
# Import necessary libraries if not already imported in previous cells
import pandas as pd
import kagglehub

# Load the dataset (redundant if 7Fc2lR1ALww5 was run, but ensures df is defined)
# 載入資料集 (如果 7Fc2lR1ALww5 儲存格已執行，此步驟為多餘，但確保 df 已定義)
path = kagglehub.dataset_download("altavish/boston-housing-dataset")
df = pd.read_csv(f"{path}/HousingData.csv")

print(df.isnull().sum()) # 檢查並顯示每個欄位的缺失值數量

# 使用中位數填補數值型欄位的缺失值
for col in df.columns:
    if df[col].isnull().any(): # 如果欄位有任何缺失值
        if df[col].dtype != 'object': # 並且該欄位是數值型的 (非物件/字串)
            df[col].fillna(df[col].median(), inplace=True) # 使用該欄位的中位數填補缺失值

print('\nAfter handling missing values:') # 顯示處理缺失值後的訊息
print(df.isnull().sum()) # 再次檢查並顯示每個欄位的缺失值數量，確認已無缺失值

Using Colab cache for faster access to the 'boston-housing-dataset' dataset.
CRIM       20
ZN         20
INDUS      20
CHAS       20
NOX         0
RM          0
AGE        20
DIS         0
RAD         0
TAX         0
PTRATIO     0
B           0
LSTAT      20
MEDV        0
dtype: int64

After handling missing values:
CRIM       0
ZN         0
INDUS      0
CHAS       0
NOX        0
RM         0
AGE        0
DIS        0
RAD        0
TAX        0
PTRATIO    0
B          0
LSTAT      0
MEDV       0
dtype: int64


/tmp/ipykernel_1548/4275897348.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True) # 使用該欄位的中位數填補缺失值


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     506 non-null    float64
 1   ZN       506 non-null    float64
 2   INDUS    506 non-null    float64
 3   CHAS     506 non-null    float64
 4   NOX      506 non-null    float64
 5   RM       506 non-null    float64
 6   AGE      506 non-null    float64
 7   DIS      506 non-null    float64
 8   RAD      506 non-null    int64  
 9   TAX      506 non-null    int64  
 10  PTRATIO  506 non-null    float64
 11  B        506 non-null    float64
 12  LSTAT    506 non-null    float64
 13  MEDV     506 non-null    float64
dtypes: float64(12), int64(2)
memory usage: 55.5 KB


In [23]:
# Separate features (X) and target (y)
# 將資料分為特徵 (X) 和目標變數 (y)
X = df.drop('MEDV', axis=1) # X 包含所有特徵，'MEDV' 欄位被移除
y = df['MEDV'] # y 包含目標變數 'MEDV' (房價)

In [24]:
from sklearn.model_selection import train_test_split # 從 sklearn 導入 train_test_split 函數

# 將數據分割成訓練集和測試集
# test_size=0.2 表示測試集佔總數據的 20%
# random_state=42 用於確保每次分割結果一致 (可重現性)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training data shape: {X_train.shape}') # 顯示訓練集的特徵數據形狀
print(f'Testing data shape: {X_test.shape}') # 顯示測試集的特徵數據形狀

Training data shape: (404, 13)
Testing data shape: (102, 13)


### 建立模型

### 效能評量

In [15]:
#start coding here

In [25]:
# Linear Regression Model (線性迴歸模型)
from sklearn.linear_model import LinearRegression # 導入線性迴歸模型
from sklearn.metrics import r2_score # 導入 R 平方分數評估指標

# 初始化並訓練線性迴歸模型
linear_model = LinearRegression() # 創建一個線性迴歸模型實例
linear_model.fit(X_train, y_train) # 使用訓練數據訓練模型

# 在訓練集和測試集上進行預測
y_train_pred_lr = linear_model.predict(X_train) # 使用訓練好的模型預測訓練集的目標值
y_test_pred_lr = linear_model.predict(X_test) # 使用訓練好的模型預測測試集的目標值

# 計算訓練集和測試集的 R 平方分數
r2_train_lr = r2_score(y_train, y_train_pred_lr) # 計算訓練集 R 平方分數
r2_test_lr = r2_score(y_test, y_test_pred_lr) # 計算測試集 R 平方分數

print("Linear Regression Model Performance:") # 打印線性迴歸模型的性能標題
print(f"  Training R^2: {r2_train_lr:.4f}") # 打印訓練集 R 平方分數，保留四位小數
print(f"  Testing R^2: {r2_test_lr:.4f}") # 打印測試集 R 平方分數，保留四位小數

Linear Regression Model Performance:
  Training R^2: 0.7426
  Testing R^2: 0.6591


In [26]:
# CART Regression Tree Model (CART 迴歸樹模型)
from sklearn.tree import DecisionTreeRegressor # 導入決策樹迴歸器

# 初始化並訓練 CART 迴歸樹模型
cart_model = DecisionTreeRegressor(random_state=42) # 創建一個決策樹迴歸器實例，設定隨機種子以確保結果可重現
cart_model.fit(X_train, y_train) # 使用訓練數據訓練模型

# 在訓練集和測試集上進行預測
y_train_pred_cart = cart_model.predict(X_train) # 使用訓練好的模型預測訓練集的目標值
y_test_pred_cart = cart_model.predict(X_test) # 使用訓練好的模型預測測試集的目標值

# 計算訓練集和測試集的 R 平方分數
r2_train_cart = r2_score(y_train, y_train_pred_cart) # 計算訓練集 R 平方分數
r2_test_cart = r2_score(y_test, y_test_pred_cart) # 計算測試集 R 平方分數

print("CART Regression Tree Model Performance:") # 打印 CART 迴歸樹模型的性能標題
print(f"  Training R^2: {r2_train_cart:.4f}") # 打印訓練集 R 平方分數，保留四位小數
print(f"  Testing R^2: {r2_test_cart:.4f}") # 打印測試集 R 平方分數，保留四位小數

CART Regression Tree Model Performance:
  Training R^2: 1.0000
  Testing R^2: 0.6810


In [27]:
# Performance Comparison (效能比較)

print("\nModel Performance Comparison (R^2 Score):") # 打印模型效能比較標題
print("------------------------------------------")
print(f"Linear Regression - Test R^2: {r2_test_lr:.4f}") # 打印線性迴歸在測試集上的 R 平方分數
print(f"CART Regression Tree - Test R^2: {r2_test_cart:.4f}") # 打印 CART 迴歸樹在測試集上的 R 平方分數

# 比較兩個模型在測試集上的 R 平方分數
if r2_test_lr > r2_test_cart:
    print("\nConclusion: Linear Regression performs better on the test set.") # 如果線性迴歸表現較好
elif r2_test_cart > r2_test_lr:
    print("\nConclusion: CART Regression Tree performs better on the test set.") # 如果 CART 迴歸樹表現較好
else:
    print("\nConclusion: Both models perform similarly on the test set.") # 如果兩個模型表現相似


Model Performance Comparison (R^2 Score):
------------------------------------------
Linear Regression - Test R^2: 0.6591
CART Regression Tree - Test R^2: 0.6810

Conclusion: CART Regression Tree performs better on the test set.


# 結論

### 結論

這次作業中，我比較了兩種預測波士頓房價的模型表現：**線性迴歸 (Linear Regression)** 和 **CART 迴歸樹 (CART Regression Tree)**。

**我的模型效能觀察 (R^2 分數)：**

*   **線性迴歸模型 (Linear Regression)**
    *   訓練資料 $R^2$: `0.7426`
    *   測試資料 $R^2$: `0.6591`

*   **CART 迴歸樹模型 (CART Regression Tree)**
    *   訓練資料 $R^2$: `1.0000`
    *   測試資料 $R^2$: `0.6810`

**我的分析與心得：**

1.  **訓練資料表現**：我發現 CART 迴歸樹在訓練資料上得到了完美的 $R^2$ (`1.0000`)，這表示它非常精準地記住了訓練資料的模式。不過，這也讓我想到它可能有**過度擬合 (Overfitting)** 的問題，也就是說，它可能對訓練資料太過「專精」，導致在沒看過的資料上表現不一定好。

2.  **測試資料表現 (泛化能力)**：對我來說，模型在測試資料上的表現更重要，因為這代表它對新資料的預測能力。
    *   線性迴歸在測試資料上的 $R^2$ 是 `0.6591`。
    *   CART 迴歸樹在測試資料上的 $R^2$ 則是 `0.6810`。

**我的最終結論：**

儘管 CART 迴歸樹在訓練資料上表現得非常完美，但在**測試資料上，它的 $R^2$ (`0.6810`) 仍然稍微比線性迴歸模型 (`0.6591`) 來得好**。這說明在這次預測波士頓房價的任務中，CART 迴歸樹對於未見過的資料，預測能力稍微佔優勢。

綜合來看，我會認為**CART 迴歸樹模型在這次分析中是比較好的選擇**。